# 26 — Shuffle, Partitioning & Skew

> **Sketch note:** read the visual model before the code. Predict first.

![Sketch note](../assets/sketch-notes/26-shuffle-partitioning-skew.svg)

## Why this exists

Large-scale systems are mechanisms for moving work, state and time under constraints. This notebook builds the mechanism before introducing a vendor.

## Learning objectives

- Build a transferable mental model.
- Predict before execution.
- Measure behavior with deterministic data.
- Break one assumption and diagnose the result.
- Connect the mechanism to production decisions.

## Prerequisites

Python basics and earlier data-engineering modules. No paid cloud service or external API is required.

## Mental model

keys → partitions → shuffle


## Capability contract

**Capability:** C06 — Streaming Lakehouse & Scale  
**Workstream:** Distributed Data Platform  
**Primary roles:** Data Engineer|Platform Engineer|Architect  
**You will prove:** Measure a scale bottleneck and choose a bounded mitigation with explicit trade-offs.


## Simulation → reality bridge

**Real-system bridge:** Spark shuffle and partitioning

**What this simulation proves:** It isolates the mechanism under study so you can predict and measure it without infrastructure noise.

**What the simulation hides:** The notebook models partition placement and skew deterministically. Real systems add network shuffle, serialization, executor memory pressure and task-level retries.

**Transfer challenge:** Construct a skewed workload and identify which evidence would distinguish key skew from slow infrastructure or downstream backpressure.

**Production boundary:** Do not describe the toy implementation as the production technology. Before shipping, identify the real-system evidence, operational controls, and failure modes that remain untested here.


In [ ]:
from collections import Counter
records=[("C001",i) for i in range(90)]+[("C002",i) for i in range(5)]+[("C003",i) for i in range(5)]
def partition_counts(rows,n):return Counter(hash(k)%n for k,_ in rows)
counts=partition_counts(records,4);print(counts)

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Predict before you run

Write the expected behavior first.

In [ ]:
prediction="one partition should be much larger than the others";prediction

## Build

Implement or inspect the smallest mechanism that answers the question.

In [ ]:
print(counts);print("max/min ratio:",max(counts.values())/min(counts.values()))

## Measure

Turn behavior into evidence. Avoid unsupported performance claims.

In [ ]:
avg=sum(counts.values())/len(counts);print("skew ratio vs average:",round(max(counts.values())/avg,2))

## Break

Introduce one controlled failure or adverse condition.

In [ ]:
salted=[(f"{k}#{i%6}",v) for i,(k,v) in enumerate(records)];print("before:",counts);print("after:",partition_counts(salted,4))

## Diagnose

Explain the difference between expected and observed behavior.

In [ ]:
print("Diagnosis: partitioning determines where work lands; a hot key can create a straggler even when total data is modest.")

## Production

Real systems add network cost, memory limits, retries, observability, security, schema contracts and operational ownership. This is a mental model, not a production guarantee.

## Explain it in 60–90 seconds

Explain the problem, mechanism, state transition and trade-off without naming a vendor first.

## Your task

**Goal:** Show how partitioning and skew change distributed work.

**Do this:** Change one assumption, predict the result, measure the distribution/work impact, and connect the finding to the paired Acme ticket.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## Challenge

Change one assumption, predict the result, then connect it to the paired Acme Commerce ticket.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S09 — DATA-1120: Event stream backlog** → [`../work_simulations/09_acme_commerce_stream_backlog/README.md`](../work_simulations/09_acme_commerce_stream_backlog/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


## Takeaways

- Mechanism first, tool second.
- Predict before execution.
- Measure what can be measured.
- Break deliberately.
- Diagnose before fixing.
- Carry the mental model into engineering work.

---

## Check your work

You have finished the exercise. **Now** compare your reasoning and evidence with the reference solution.

[Open the reference solution for Notebook 26](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/26.md)
